# Governing with Artificial Intelligence
## Mapping the Knowledge Systems Shaping Urban Intelligence

**Desmond Lartey · Kris M.Y. Law**
*Technology in Society* **86** (2026) 103321 · [10.1016/j.techsoc.2026.103321](https://doi.org/10.1016/j.techsoc.2026.103321)

---

### How to use this notebook

The analysis runs in **ten numbered stages**. Every stage:

- states what it does and which part of the paper it corresponds to,
- prints what it produced so you can check it before moving on,
- **saves a checkpoint** to `outputs/checkpoints/`.

Because of the checkpoints you do not have to run the notebook top to bottom.
Run **Stage 0** and **Stage 1** (they are cheap and define everything), then jump
to whichever stage you care about and run its *Resume* cell first.

### Two data modes

| Mode | What it uses | What you get |
|---|---|---|
| `demo` *(default)* | A synthetic corpus generated from the paper's published marginals | A fully runnable pipeline, figures clearly stamped **DEMO** |
| `real` | Your own corpus in `data/` | The actual analysis |

> **The demo corpus is not the study's data.** It is generated so that the code is
> runnable and inspectable without waiting on a data deposit. Numbers and figures
> produced in `demo` mode do not reproduce the paper and every figure is stamped to
> say so. Set `CONFIG["data_mode"] = "real"` and drop your corpus into `data/` to
> run the real thing.

### Requirements

`pip install -r requirements.txt`. Everything works offline except
`sentence-transformers`, which downloads a ~90 MB model on first use. If it is not
installed the notebook automatically falls back to a TF-IDF backend so nothing
breaks — Stage 3 explains the trade-off.

---
# Stage 0 — Setup

Imports, random seeds, folders, and the single `CONFIG` dictionary that controls
every choice made downstream. Change settings **here**, not scattered through the
notebook, so that any run can be described by one printed block.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------- CONFIG
CONFIG = {
    # "demo" = synthetic corpus from published marginals; "real" = your own data
    "data_mode":        "demo",

    "data_dir":         "../data",
    "output_dir":       "../outputs",

    # File to read in real mode. CSV or XLSX both work.
    "corpus_file":      "corpus.csv",

    # Semantic backend: "auto" | "sbert" | "tfidf"
    "backend":          "auto",
    "sbert_model":      "all-MiniLM-L6-v2",
    "batch_size":       256,

    # Lens assignment
    "threshold":        0.45,      # paper's cosine gate (SBERT scale)
    "threshold_on":     "cosine",  # "cosine" or "share"
    "keyword_weight":   1.0,       # set to 0.0 for a purely semantic assignment
    "semantic_weight":  1.0,

    # Validation / inference
    "validation_frac":  0.10,
    "n_permutations":   9999,
    "n_bootstrap":      2000,
    "permanova_max_n":  800,
    "permanova_repeats": 5,

    "seed":             42,
    "dpi":              200,
}

SEED = CONFIG["seed"]
np.random.seed(SEED)

def _repo_relative(p):
    """Resolve a repo-relative path whether the kernel started in notebooks/ or the
    repository root, so the notebook runs from either."""
    p = Path(p)
    if p.exists():
        return p
    alt = Path(str(p).replace("../", "", 1))
    return alt if (alt.exists() or Path("notebooks").is_dir()) else p


DATA_DIR = _repo_relative(CONFIG["data_dir"])
OUT_DIR = _repo_relative(CONFIG["output_dir"])
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
CKPT_DIR = OUT_DIR / "checkpoints"
for d in (DATA_DIR, OUT_DIR, FIG_DIR, TAB_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": CONFIG["dpi"],
                     "savefig.bbox": "tight", "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold"})

print("Configuration")
print("-" * 60)
for k, v in CONFIG.items():
    print(f"  {k:20s} {v}")
print("-" * 60)
print(f"  outputs -> {OUT_DIR.resolve()}")

In [ ]:
def save_checkpoint(name: str, obj, index: bool = True) -> Path:
    """Persist a stage result so later stages can be run independently."""
    path = CKPT_DIR / f"{name}.csv"
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=index)
    elif isinstance(obj, pd.Series):
        obj.to_frame("value").to_csv(path, index=index)
    else:
        path = CKPT_DIR / f"{name}.json"
        path.write_text(json.dumps(obj, indent=2, default=str))
    print(f"  checkpoint saved: {path.name}")
    return path


def load_checkpoint(name: str, index_col=0):
    """Reload a stage result. Raises a readable error if the stage never ran."""
    csv, js = CKPT_DIR / f"{name}.csv", CKPT_DIR / f"{name}.json"
    if csv.exists():
        return pd.read_csv(csv, index_col=index_col)
    if js.exists():
        return json.loads(js.read_text())
    raise FileNotFoundError(
        f"No checkpoint '{name}' in {CKPT_DIR}. "
        f"Run the stage that produces it, then come back to this cell.")


def environment_report() -> dict:
    import sklearn, scipy, matplotlib
    mods = {"python": sys.version.split()[0], "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
            "seaborn": sns.__version__, "networkx": nx.__version__}
    for name in ("statsmodels", "sentence_transformers"):
        try:
            mods[name] = __import__(name).__version__
        except Exception:
            mods[name] = "not installed"
    mods["platform"] = platform.platform()
    mods["run_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return mods


ENV = environment_report()
print("Environment")
print("-" * 60)
for k, v in ENV.items():
    print(f"  {k:22s} {v}")
save_checkpoint("stage0_environment", ENV)

---
# Stage 1 — The conceptual framework

Everything the analysis depends on conceptually is declared here: the 12 search
indicators, the 3 lenses, the a-priori indicator→lens map (Table 2), and the values
published in the paper's figures.

**The published values are kept as a reference target, not as figure input.** Every
figure in this notebook is computed from whatever corpus you load. The published
numbers are used in one place only — Stage 5, where they are correlated against your
rebuilt matrix so you can see how closely your data tracks the paper's.

In [ ]:
INDICATORS = ["Action", "Agency", "Culture", "Data", "Governance", "Materiality",
              "Personality", "Security", "Space", "Sustainability", "Technology", "Time"]

LENSES = ["Key Stakeholders and entities",
          "Models of Interaction",
          "External influencing factors"]

# Table 2 in the paper
INDICATOR_TO_LENS = {
    "Technology":     "Key Stakeholders and entities",
    "Materiality":    "Key Stakeholders and entities",
    "Sustainability": "Key Stakeholders and entities",
    "Action":         "Models of Interaction",
    "Space":          "Models of Interaction",
    "Culture":        "Models of Interaction",
    "Agency":         "External influencing factors",
    "Data":           "External influencing factors",
    "Governance":     "External influencing factors",
    "Security":       "External influencing factors",
    "Personality":    "External influencing factors",
    "Time":           "External influencing factors",
}

# Reference vectors for the semantic stage (Section 3.1.2)
LENS_DESCRIPTIONS = {
    "Key Stakeholders and entities":
        "Infrastructure, sensors, IoT, AIoT, hardware, data centres, platforms, "
        "institutions and organisations that enable artificial intelligence in cities.",
    "Models of Interaction":
        "Interaction, decision-making, feedback loops, co-agency, predictive analytics, "
        "and how AI systems and urban actors act on one another.",
    "External influencing factors":
        "Governance, regulation, policy, ethics, accountability, legitimacy, security, "
        "sustainability and cultural context shaping AI in cities.",
}

LENS_COLORS = {
    "Key Stakeholders and entities": "#C0392B",
    "Models of Interaction":         "#2471A3",
    "External influencing factors":  "#1E8449",
}
LENS_SHORT = {"Key Stakeholders and entities": "Stakeholders",
              "Models of Interaction": "Interaction",
              "External influencing factors": "External"}

pd.DataFrame({"SearchIndicator": INDICATORS,
              "Assigned lens": [INDICATOR_TO_LENS[i] for i in INDICATORS]})

In [ ]:
# ---- Published values from the paper, kept ONLY as a comparison target ----

# Figure 3b: weighted semantic matches, indicator x lens
PAPER_LENS_WEIGHTS = pd.DataFrame({
    "Key Stakeholders and entities": [3.184, 11.406, 11.814, 34.658, 17.654, 5.784,
                                      3.661, 2.402, 22.788, 54.695, 59.124, 4.803],
    "Models of Interaction":         [5.166, 2.225, 11.035, 16.032, 2.046, 4.252,
                                      1.122, 0.585, 8.165, 17.330, 19.398, 2.679],
    "External influencing factors":  [1.179, 35.071, 9.036, 29.109, 52.921, 4.241,
                                      6.406, 1.171, 10.369, 24.763, 17.346, 2.824],
}, index=pd.Index(INDICATORS, name="SearchIndicator"))

# Figure 1 inset: article frequency per indicator per lens
PAPER_FREQ = pd.DataFrame({
    "Key Stakeholders and entities": [404, 101, 43, 230, 191, 7, 186, 6, 267, 350, 646, 24],
    "Models of Interaction":         [234, 172, 36, 124, 116, 9, 268, 2, 276, 253, 223, 13],
    "External influencing factors":  [138, 174, 30, 203, 221, 5, 110, 5, 249, 320, 165, 13],
}, index=pd.Index(INDICATORS, name="SearchIndicator"))

# Figure 4b: weighted lens contribution per year
PAPER_TEMPORAL = pd.DataFrame({
    "Year": [1990, 2006, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "Key Stakeholders and entities": [0, 0, 18, 49, 80, 84, 147, 195, 282, 338, 393, 805, 205],
    "Models of Interaction":         [1, 1, 34, 50, 73, 102, 137, 180, 269, 374, 434, 903, 265],
    "External influencing factors":  [0, 0, 10, 22, 46, 47, 56, 82, 169, 194, 266, 586, 167],
}).set_index("Year")

# Figure 6: which indicators feed which governance subsystem
SUBSYSTEM_INDICATORS = {
    "Decision-Making Dynamics":     ["Action", "Agency", "Governance"],
    "Citizen Participation Models": ["Culture", "Agency", "Governance"],
    "Human-AI Interfaces":          ["Personality", "Agency", "Action"],
    "Urban Data Infrastructure":    ["Data", "Technology", "Materiality"],
    "AI Hardware & Platforms":      ["Technology", "Materiality", "Security"],
    "Institutional Stakeholders":   ["Governance", "Technology", "Sustainability"],
    "Platform Governance":          ["Governance", "Data", "Technology"],
    "Ethical Standards":            ["Governance", "Data", "Culture"],
    "Accountability Mechanisms":    ["Governance", "Agency", "Security"],
    "Legal/Policy Instruments":     ["Governance", "Time", "Security"],
    "Feedback Loops":               ["Action", "Time", "Agency"],
    "Public Legitimacy":            ["Culture", "Governance", "Personality"],
}
SUBSYSTEMS = list(SUBSYSTEM_INDICATORS)

print(f"{len(INDICATORS)} indicators, {len(LENSES)} lenses, {len(SUBSYSTEMS)} subsystems")
print("Reference tables loaded (used for comparison in Stage 5, not as figure input).")

---
# Stage 2 — Load the corpus

### If you have the real data

Put a CSV or XLSX at `data/corpus.csv` with these columns and set
`CONFIG["data_mode"] = "real"`:

| Column | Type | Notes |
|---|---|---|
| `Title` | text | required |
| `Abstract` | text | required; may be blank for some rows |
| `Keyword` | text | required; semicolon separated |
| `Year` | integer | required, 1900–2100 |
| `SearchIndicator` | text | required, one of the 12 names in Stage 1 |
| `DOI` | text | optional but recommended |

The validator below checks all of this and tells you exactly what is wrong rather
than failing several stages later with an unrelated error.

### If you do not

The default `demo` mode builds a synthetic corpus of the same shape from the paper's
published marginals. It exists so the code can be read and executed today. It is not
the study's data and will not reproduce the study's numbers.

In [ ]:
REQUIRED_COLUMNS = ["Title", "Abstract", "Keyword", "Year", "SearchIndicator"]

DEMO_VOCAB = {
    "Technology":     ["AIoT", "machine learning", "sensor network", "edge computing",
                       "autonomous infrastructure", "urban platform"],
    "Action":         ["predictive analytics", "algorithmic action", "urban intervention",
                       "real-time response", "decision support"],
    "Space":          ["digital twin", "spatial governance", "urban morphology",
                       "smart urban space", "public space"],
    "Data":           ["datafication", "data ethics", "surveillance", "algorithmic bias",
                       "open data", "data governance"],
    "Sustainability": ["climate-smart city", "urban decarbonisation", "resilience",
                       "sustainable development goals", "energy efficiency"],
    "Security":       ["predictive policing", "cybersecurity", "AI surveillance",
                       "risk mitigation", "critical infrastructure"],
    "Agency":         ["distributed agency", "human-AI interaction", "AI as actor",
                       "sociotechnical system", "machine autonomy"],
    "Culture":        ["AI imaginaries", "AI narratives", "cultural urbanism",
                       "future cities discourse", "public perception"],
    "Personality":    ["affective computing", "chatbot", "human-centred AI",
                       "emotional AI", "anthropomorphism"],
    "Governance":     ["AI regulation", "smart city governance", "AI ethics",
                       "urban planning policy", "accountability"],
    "Time":           ["temporal governance", "urban foresight", "anticipatory planning",
                       "planning horizon", "future-oriented governance"],
    "Materiality":    ["data centre", "urban hardware", "smart grid",
                       "material politics of AI", "physical infrastructure"],
}


def synthesize_corpus(n_articles: int = 5634, seed: int = SEED) -> pd.DataFrame:
    """Stand-in corpus drawn from the paper's PUBLISHED marginals.

    Indicator mix comes from Figure 1's frequency table and the year distribution
    from Figure 4b, so the corpus has a realistic shape. The text is generated, so
    the semantic stage has something to work on. This is NOT the study's data.
    """
    rng = np.random.default_rng(seed)
    ind_p = (PAPER_FREQ.sum(axis=1) / PAPER_FREQ.sum().sum())
    yr_p = (PAPER_TEMPORAL.sum(axis=1) / PAPER_TEMPORAL.sum().sum())

    inds = rng.choice(ind_p.index.to_numpy(), size=n_articles, p=ind_p.to_numpy())
    yrs = rng.choice(yr_p.index.to_numpy(), size=n_articles, p=yr_p.to_numpy())

    rows = []
    for i, (ind, yr) in enumerate(zip(inds, yrs)):
        t = rng.choice(DEMO_VOCAB[ind], size=3, replace=False)
        rows.append({
            "ArticleID": f"SYN-{i:05d}",
            "Title": f"{t[0].capitalize()} and {t[1]}: evidence on {t[2]} in cities ({i:05d})",
            "Abstract": (f"This study examines {t[0]} in the context of {t[1]} for urban "
                         f"systems. Drawing on {t[2]}, it considers how artificial "
                         f"intelligence reshapes urban governance and decision-making."),
            "Keyword": "; ".join(t),
            "Year": int(yr),
            "SearchIndicator": ind,
        })
    return pd.DataFrame(rows)


def validate_corpus(df: pd.DataFrame, strict: bool = True):
    """Schema and content checks. Returns (blocking_problems, warnings)."""
    problems, notes = [], []
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        problems.append(f"Missing required column(s): {missing}. Found: {list(df.columns)}")
        return problems, notes

    yr = pd.to_numeric(df["Year"], errors="coerce")
    if int(yr.isna().sum()):
        notes.append(f"{int(yr.isna().sum())} row(s) have an unparseable Year (dropped later).")
    odd = (~yr.between(1900, 2100)) & yr.notna()
    if odd.any():
        notes.append(f"{int(odd.sum())} row(s) have Year outside 1900-2100.")

    found = set(df["SearchIndicator"].dropna().astype(str).str.strip())
    unknown = found - set(INDICATOR_TO_LENS)
    if unknown:
        msg = (f"Unrecognised SearchIndicator value(s): {sorted(unknown)}. "
               f"Expected one of {INDICATORS}.")
        (problems if strict else notes).append(msg)
    absent = set(INDICATOR_TO_LENS) - found
    if absent:
        notes.append(f"No articles for indicator(s): {sorted(absent)}")

    blank = df["Abstract"].astype(str).str.strip().eq("").mean()
    if blank > 0.5:
        notes.append(f"{blank:.0%} of abstracts are blank; the semantic stage will be weak.")
    dupes = int(df.duplicated(subset=["Title"]).sum())
    if dupes:
        notes.append(f"{dupes} duplicate Title value(s).")
    return problems, notes

In [ ]:
if CONFIG["data_mode"] == "demo":
    print("!" * 70)
    print("DEMO MODE - synthetic corpus generated from the paper's published marginals.")
    print("This is NOT the study's data. Figures will be stamped accordingly.")
    print("Set CONFIG['data_mode'] = 'real' once data/corpus.csv is in place.")
    print("!" * 70)
    corpus = synthesize_corpus()
else:
    path = DATA_DIR / CONFIG["corpus_file"]
    if not path.exists():
        raise FileNotFoundError(
            f"{path.resolve()} not found.\n"
            f"Place your corpus there (see the schema table above), "
            f"or set CONFIG['data_mode'] = 'demo'.")
    corpus = (pd.read_excel(path, dtype=str) if path.suffix.lower() in (".xlsx", ".xls")
              else pd.read_csv(path, dtype=str))

corpus.columns = corpus.columns.str.strip()
for c in REQUIRED_COLUMNS:
    if c in corpus.columns:
        corpus[c] = corpus[c].fillna("").astype(str).str.strip()

problems, notes = validate_corpus(corpus)
print(f"\nRows: {len(corpus):,}   Columns: {list(corpus.columns)}")
if notes:
    print("\nWarnings:")
    for n in notes:
        print("  -", n)
if problems:
    print("\nBLOCKING PROBLEMS:")
    for p in problems:
        print("  -", p)
    raise ValueError("Corpus failed validation. Fix the problems above and re-run.")
print("\nValidation passed.")

corpus["Year"] = pd.to_numeric(corpus["Year"], errors="coerce")
corpus = corpus.dropna(subset=["Year"]).copy()
corpus["Year"] = corpus["Year"].astype(int)
corpus = corpus.reset_index(drop=True)

save_checkpoint("stage2_corpus", corpus, index=False)
display(corpus.head(3))
display(corpus["SearchIndicator"].value_counts().rename("articles").to_frame())

---
# Stage 3 — Semantic scoring

Each article's `Title + Abstract + Keyword` is embedded and compared by cosine
similarity to the three lens descriptions from Stage 1 (paper Section 3.1.2).

### Choosing a backend

Encoding is batched, so a corpus of a few thousand abstracts takes roughly twenty
seconds on CPU rather than several minutes.

`sentence-transformers` pulls a ~90 MB model on first use, which is awkward on a
metered connection and impossible on an air-gapped machine. If it is unavailable the
notebook falls back to TF-IDF cosine so nothing blocks.

| Backend | Matches the paper | Needs a download | Deterministic |
|---|---|---|---|
| `sbert` | yes | yes, ~90 MB | yes |
| `tfidf` | no — an approximation | no | yes |

TF-IDF cosines sit on a much lower scale than SBERT's, so the paper's 0.45 gate does
not transfer. The cell below detects the backend and adjusts the gate, and prints
which rule it used.

In [ ]:
def corpus_text(df: pd.DataFrame) -> pd.Series:
    return (df["Title"].astype(str) + ". " + df["Abstract"].astype(str) + ". "
            + df["Keyword"].astype(str)).str.strip()


def semantic_lens_scores(df, backend="auto", model_name="all-MiniLM-L6-v2",
                         batch_size=256, verbose=True):
    """Cosine similarity of every article against the three lens descriptions.

    Returns (scores_dataframe, backend_actually_used).
    """
    text = corpus_text(df)

    if backend in ("auto", "sbert"):
        try:
            from sentence_transformers import SentenceTransformer
            if verbose:
                print(f"Encoding {len(text):,} articles with {model_name} (batched)...")
            model = SentenceTransformer(model_name)
            emb = model.encode(text.tolist(), batch_size=batch_size,
                               convert_to_numpy=True, normalize_embeddings=True,
                               show_progress_bar=verbose)
            ref = model.encode([LENS_DESCRIPTIONS[l] for l in LENSES],
                               convert_to_numpy=True, normalize_embeddings=True)
            return pd.DataFrame(emb @ ref.T, columns=LENSES, index=df.index), "sbert"
        except ImportError:
            if backend == "sbert":
                raise ImportError(
                    "backend='sbert' requires sentence-transformers.\n"
                    "  pip install sentence-transformers\n"
                    "or set CONFIG['backend'] = 'tfidf'.")
            if verbose:
                print("sentence-transformers not installed -> using TF-IDF backend.")

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True,
                          ngram_range=(1, 2), min_df=2)
    X = vec.fit_transform(text.tolist())
    R = vec.transform([LENS_DESCRIPTIONS[l] for l in LENSES])
    return pd.DataFrame((normalize(X) @ normalize(R).T).toarray(),
                        columns=LENSES, index=df.index), "tfidf"

In [ ]:
sims, BACKEND = semantic_lens_scores(
    corpus, backend=CONFIG["backend"],
    model_name=CONFIG["sbert_model"], batch_size=CONFIG["batch_size"])

# The paper's 0.45 cosine gate is calibrated to SBERT. TF-IDF cosines are far
# smaller, so gate on the share of similarity mass instead of the raw value.
if BACKEND == "tfidf" and CONFIG["threshold_on"] == "cosine":
    CONFIG["threshold_on"] = "share"
    CONFIG["threshold"] = 0.40
    print("\nBackend is TF-IDF: switched the gate to share >= 0.40 "
          "(the 0.45 cosine rule is SBERT-specific).")

print(f"\nBackend used: {BACKEND}")
print(f"Gate: {CONFIG['threshold_on']} >= {CONFIG['threshold']}")
print(f"\nSimilarity distribution per lens:")
display(sims.describe().T[["mean", "std", "min", "50%", "max"]].round(3))

save_checkpoint("stage3_similarities", sims)

---
# Stage 4 — Lens assignment

Paper Section 3.1.3 describes a **top-2 ensemble**: the keyword signal and the two
strongest semantic matches are merged into one score per lens.

One property of this design is worth stating plainly, because it shapes how Stages
5–8 are built:

> **With `keyword_weight >= 1`, an article's lens is fully determined by its
> `SearchIndicator`.** The assignment is a lookup, not a classification. The kappa in
> Stage 6 then measures whether a human agrees with the lookup table — which is a
> reasonable thing to measure, but it is not a measure of model accuracy.

The scoring below is explicit so you can see and change this:

```
score[lens] = keyword_weight · 1{indicator maps to lens}
            + semantic_weight · share[lens]   (top-2 only, above the gate)
```

`share` is a lens's fraction of the article's total similarity, so the semantic term
stays on a 0–1 scale whichever backend produced it. Set `keyword_weight = 0` for a
purely semantic assignment; the sensitivity cell does exactly that so you can see how
far the two signals agree.

In [ ]:
def keyword_lens(df: pd.DataFrame) -> pd.Series:
    return df["SearchIndicator"].astype(str).str.strip().map(INDICATOR_TO_LENS)


def ensemble_lens(df, sims, threshold=0.45, threshold_on="cosine",
                  keyword_weight=1.0, semantic_weight=1.0):
    """Top-2 semantic ensemble merged with the keyword signal (Section 3.1.3)."""
    kw = keyword_lens(df)
    score = pd.DataFrame(0.0, index=df.index, columns=LENSES)
    for lens in LENSES:
        score[lens] += (kw == lens).astype(float) * keyword_weight

    if sims is not None and semantic_weight:
        raw = sims[LENSES].to_numpy(dtype=float)
        total = raw.sum(axis=1, keepdims=True)
        share = np.divide(raw, total, out=np.zeros_like(raw), where=total > 0)
        gate = raw if threshold_on == "cosine" else share

        top2 = np.argsort(-raw, axis=1)[:, :2]
        rows = np.repeat(np.arange(raw.shape[0]), 2)
        cols = top2.ravel()
        keep = gate[rows, cols] >= threshold

        contrib = np.zeros_like(raw)
        contrib[rows[keep], cols[keep]] = share[rows[keep], cols[keep]]
        score += contrib * semantic_weight

    winner = score.idxmax(axis=1)
    winner[score.max(axis=1) <= 0] = "Unclassified"
    return winner, score


corpus["Keyword_Lens"] = keyword_lens(corpus)
corpus["Semantic_Lens"] = sims[LENSES].idxmax(axis=1)
corpus["ConceptualLens"], lens_scores = ensemble_lens(
    corpus, sims,
    threshold=CONFIG["threshold"], threshold_on=CONFIG["threshold_on"],
    keyword_weight=CONFIG["keyword_weight"], semantic_weight=CONFIG["semantic_weight"])

agree = (corpus["Keyword_Lens"] == corpus["Semantic_Lens"]).mean()
print(f"Keyword and semantic signals agree on {agree:.1%} of articles "
      f"({int(agree * len(corpus)):,} of {len(corpus):,}).")
print("\nFinal lens distribution:")
display(corpus["ConceptualLens"].value_counts().rename("articles").to_frame())

save_checkpoint("stage4_tagged", corpus, index=False)

In [ ]:
# --- Sensitivity: how much does the ensemble actually depend on its settings? ---
rows = []
for kw_w in (0.0, 0.5, 1.0):
    for thr in (0.30, 0.40, 0.45, 0.55):
        w, _ = ensemble_lens(corpus, sims, threshold=thr,
                             threshold_on=CONFIG["threshold_on"],
                             keyword_weight=kw_w, semantic_weight=1.0)
        rows.append({"keyword_weight": kw_w, "threshold": thr,
                     "agreement_with_baseline": (w == corpus["ConceptualLens"]).mean(),
                     **w.value_counts(normalize=True).reindex(LENSES).fillna(0)
                       .rename(LENS_SHORT).to_dict()})

sensitivity = pd.DataFrame(rows)
save_checkpoint("stage4_sensitivity", sensitivity, index=False)
print("Share of articles per lens under different ensemble settings.")
print("keyword_weight = 0 is the purely semantic assignment.\n")
display(sensitivity.style.format({c: "{:.1%}" for c in sensitivity.columns[2:]})
        .background_gradient(subset=["agreement_with_baseline"], cmap="Blues"))

---
# Stage 5 — Build the matrices

Two tables feed everything downstream.

### The weighted matrix is a semantic mass, not a count

This distinction matters, so it is worth being explicit about it before you change
anything downstream.

Counting articles per indicator per lens does not work here. Because each indicator
maps to exactly one lens a priori, a count pivot is a **diagonal matrix** — eleven of
twelve cells in every row are zero. PCA, NMDS and PERMANOVA on a diagonal matrix are
uninformative: they recover the lookup table you fed in and nothing else. The count
pivot is still printed below so you can see this for yourself.

Figure 3b is a heatmap of *weighted semantic matches*, and every indicator in it
carries weight on all three lenses. The quantity is therefore the **average share of
similarity** each indicator's articles place on each lens. That is what is built
below, and it produces the dense structure the figure shows.

### The trend matrix

Year × lens. Available as counts of dominant-lens articles or as summed semantic
share; the weighted version is the default because it does not throw away articles
that sit between two lenses.

In [ ]:
def build_weighted_matrix(df, sims, normalize="none", scale=100.0):
    """Indicator x lens matrix of aggregate SEMANTIC WEIGHT (not article counts)."""
    raw = sims[LENSES].to_numpy(dtype=float)
    total = raw.sum(axis=1, keepdims=True)
    share = pd.DataFrame(np.divide(raw, total, out=np.zeros_like(raw), where=total > 0),
                         columns=LENSES, index=df.index)
    share["SearchIndicator"] = df["SearchIndicator"].astype(str).str.strip().values

    m = share.groupby("SearchIndicator")[LENSES].mean() * scale
    m = m.reindex(INDICATORS).fillna(0.0)
    m.index.name = "SearchIndicator"

    if normalize == "row_percent":
        m = m.div(m.sum(axis=1), axis=0) * 100
    elif normalize == "total_percent":
        m = m / m.to_numpy().sum() * 100
    return m


def build_trend_matrix(df, sims=None, how="weight"):
    """Year x lens. how='count' tallies dominant lens; how='weight' sums share."""
    d = df.dropna(subset=["Year"]).copy()
    d["Year"] = d["Year"].astype(int)
    if how == "count":
        t = d.pivot_table(index="Year", columns="ConceptualLens", aggfunc="size", fill_value=0)
    else:
        raw = sims.loc[d.index, LENSES].to_numpy(dtype=float)
        tot = raw.sum(axis=1, keepdims=True)
        share = pd.DataFrame(np.divide(raw, tot, out=np.zeros_like(raw), where=tot > 0),
                             columns=LENSES, index=d.index)
        share["Year"] = d["Year"].values
        t = share.groupby("Year")[LENSES].sum()
    return t.reindex(columns=LENSES, fill_value=0).sort_index()


def compare_to_paper(matrix, reference=None):
    from scipy.stats import pearsonr, spearmanr
    ref = PAPER_LENS_WEIGHTS if reference is None else reference
    a = matrix.reindex(index=ref.index, columns=ref.columns).to_numpy().ravel()
    b = ref.to_numpy().ravel()
    return {"pearson_r": round(float(pearsonr(a, b)[0]), 4),
            "spearman_rho": round(float(spearmanr(a, b)[0]), 4),
            "n_cells": int(len(a))}


weighted_matrix = build_weighted_matrix(corpus, sims)
trend_matrix = build_trend_matrix(corpus, sims, how="weight")
count_matrix = corpus.pivot_table(index="SearchIndicator", columns="ConceptualLens",
                                  aggfunc="size", fill_value=0) \
                     .reindex(index=INDICATORS, columns=LENSES, fill_value=0)

print("Weighted matrix (semantic mass, indicator x lens):")
display(weighted_matrix.round(2))

dense = int((count_matrix > 0).sum(axis=1).gt(1).sum())
print(f"\nFor contrast, the article-COUNT pivot has {dense} of 12 indicators "
      f"spread across more than one lens:")
display(count_matrix)

save_checkpoint("stage5_weighted_matrix", weighted_matrix)
save_checkpoint("stage5_trend_matrix", trend_matrix)
save_checkpoint("stage5_count_matrix", count_matrix)

In [ ]:
fit = compare_to_paper(weighted_matrix)
save_checkpoint("stage5_paper_fit", fit)

print("Rebuilt matrix vs the published Figure 3b values")
print("-" * 55)
for k, v in fit.items():
    print(f"  {k:16s} {v}")

if CONFIG["data_mode"] == "demo":
    print("\n  Expect a weak correlation here: the demo corpus is synthetic text,")
    print("  so there is no reason for it to land on the paper's numbers. On the")
    print("  real corpus this is the number that tells you the rebuild worked.")
else:
    r = fit["pearson_r"]
    verdict = ("close" if r > 0.9 else "reasonable" if r > 0.7 else
               "weak - check the backend, the gate, and the lens descriptions")
    print(f"\n  Agreement with the published matrix: {verdict}")

---
# Stage 6 — Validation

Paper Section 3.1.3 reports **95.74% exact agreement** and **Cohen's κ = 0.933** on a
stratified 10% sample (n = 563).

The cell below draws the stratified sample and writes a coder workbook with the
allowed lens values, the coding rule and a blinding instruction on its own sheet, so a
second coder can work from the file alone.

κ is reported with a bootstrap 95% confidence interval over 2,000 resamples. A point
estimate on a few hundred rows is hard to read without one.

Keep Stage 4 in mind when interpreting it: with `keyword_weight >= 1` the automated
label is a deterministic function of `SearchIndicator`, so κ measures agreement
between the human coder and the a-priori lookup table rather than the accuracy of a
classifier. Set `keyword_weight = 0` in Stage 0 if you want the latter.

Once a sheet is coded, save it to `data/validation_completed.csv` and the scoring cell
will pick it up.

In [ ]:
def create_validation_sample(df, frac=0.10, seed=SEED):
    """Stratified sample by SearchIndicator, with coding instructions embedded."""
    parts = []
    for indicator, group in df.groupby("SearchIndicator"):
        n = max(1, round(len(group) * frac))
        parts.append(group.sample(n=n, random_state=seed))
    sample = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    keep = [c for c in ["ArticleID", "Title", "Abstract", "Keyword", "Year",
                        "SearchIndicator", "ConceptualLens"] if c in sample.columns]
    sample = sample[keep].rename(columns={"ConceptualLens": "Auto_Lens"})
    sample["Manual_Lens"] = ""
    sample["Coder_Confidence"] = ""
    sample["Notes"] = ""

    instructions = pd.DataFrame({
        "field": ["TASK", "Manual_Lens", "Coder_Confidence", "RULE", "BLINDING"],
        "instruction": [
            "Read Title, Abstract and Keyword only. Assign ONE lens per row.",
            f"Exactly one of: {' | '.join(LENSES)}",
            "high / medium / low",
            "Assign the lens that best captures the article's primary concern. "
            "If two fit, choose the one the article's research question centres on.",
            "Do not read Auto_Lens before coding. Hide that column while you work.",
        ]})
    return sample, instructions


validation_sample, coder_instructions = create_validation_sample(
    corpus, frac=CONFIG["validation_frac"])

vpath = TAB_DIR / "validation_sample.xlsx"
try:
    with pd.ExcelWriter(vpath) as xl:
        coder_instructions.to_excel(xl, sheet_name="INSTRUCTIONS", index=False)
        validation_sample.to_excel(xl, sheet_name="coding", index=False)
    print(f"Coder workbook written: {vpath}")
except Exception as exc:
    validation_sample.to_csv(TAB_DIR / "validation_sample.csv", index=False)
    print(f"(Excel writer unavailable: {exc}) -> wrote validation_sample.csv instead")

print(f"Stratified {CONFIG['validation_frac']:.0%} sample: {len(validation_sample):,} rows")
print(f"Paper's sample size for comparison: 563")
save_checkpoint("stage6_validation_sample", validation_sample, index=False)
display(coder_instructions)

In [ ]:
def bootstrap_kappa(y_auto, y_manual, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    a, m = np.asarray(y_auto), np.asarray(y_manual)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(a), len(a))
        if len(np.unique(m[idx])) < 2 or len(np.unique(a[idx])) < 2:
            continue
        stats.append(cohen_kappa_score(a[idx], m[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(cohen_kappa_score(a, m)), float(lo), float(hi), len(stats)


def score_validation(path):
    p = Path(path)
    if not p.exists():
        print(f"No completed sheet at {p}.")
        print("Fill in Manual_Lens in the workbook above, save it there, then re-run.")
        return None
    df = (pd.read_excel(p) if p.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(p))
    df = df.fillna("")
    coded = df[df["Manual_Lens"].astype(str).str.strip() != ""].copy()
    if coded.empty:
        print("The sheet exists but Manual_Lens is empty.")
        return None

    auto = coded["Auto_Lens"].astype(str).str.strip()
    man = coded["Manual_Lens"].astype(str).str.strip()

    bad = set(man) - set(LENSES)
    if bad:
        print(f"WARNING: unrecognised Manual_Lens value(s) {sorted(bad)} - rows dropped.")
        keep = man.isin(LENSES)
        auto, man = auto[keep], man[keep]

    agreement = (auto == man).mean() * 100
    k, lo, hi, n_ok = bootstrap_kappa(auto, man, n_boot=CONFIG["n_bootstrap"])

    print(f"Rows coded:      {len(auto):,}")
    print(f"Exact agreement: {agreement:.2f}%   (paper: 95.74%)")
    print(f"Cohen's kappa:   {k:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  "
          f"({n_ok:,} usable resamples)   (paper: 0.933)")
    print("\nPer-lens performance:")
    print(classification_report(man, auto, zero_division=0))

    cm = pd.DataFrame(confusion_matrix(man, auto, labels=LENSES),
                      index=[f"human: {LENS_SHORT[l]}" for l in LENSES],
                      columns=[f"auto: {LENS_SHORT[l]}" for l in LENSES])
    display(cm)
    result = {"n": int(len(auto)), "exact_agreement_pct": round(float(agreement), 2),
              "kappa": round(k, 4), "kappa_ci_low": round(lo, 4),
              "kappa_ci_high": round(hi, 4)}
    save_checkpoint("stage6_validation_scores", result)
    return result


validation_scores = score_validation(DATA_DIR / "validation_completed.csv")

---
# Stage 7 — Ordination

PCA, NMDS and GNMDS on the weighted matrix (paper Section 3.2.2).

**On the stress value.** `MDS.stress_` reports raw stress, whose magnitude depends on
the scale of the input and so is not comparable across runs or against published
thresholds. Kruskal's stress-1 is computed here instead, via an isotonic fit, so the
usual reading applies (Clarke 1993: <0.05 excellent, <0.10 good, <0.20 usable).

**On scikit-learn versions.** `MDS`'s `dissimilarity` and `metric` arguments are being
renamed and the default `init` is changing in 1.10. `run_mds()` inspects the signature
and passes whichever names the installed version expects, so the ordination does not
change silently when you upgrade.

**On sample size.** The ordination places 12 points in 2 dimensions. Stress will be
low almost regardless of what the data says, because two dimensions can nearly always
fit twelve points. Read it as a sanity check, not as evidence.

In [ ]:
def run_mds(distances, metric=False, seed=SEED, n_init=10, max_iter=3000):
    """MDS wrapper that survives scikit-learn's API rename."""
    import inspect
    from sklearn.manifold import MDS
    params = inspect.signature(MDS.__init__).parameters
    kw = dict(n_components=2, random_state=seed, n_init=n_init, max_iter=max_iter)
    if "metric_mds" in params:
        kw["metric"], kw["metric_mds"] = "precomputed", metric
    else:
        kw["dissimilarity"], kw["metric"] = "precomputed", metric
    if "init" in params:
        kw["init"] = "random"
    return MDS(**kw).fit_transform(np.asarray(distances))


def kruskal_stress1(orig_dist, coords):
    """Kruskal stress-1 via isotonic regression. Scale free, version independent."""
    from sklearn.isotonic import IsotonicRegression
    from scipy.spatial.distance import pdist
    iu = np.triu_indices_from(np.asarray(orig_dist), k=1)
    d_o, d_e = np.asarray(orig_dist)[iu], pdist(coords)
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(d_o, d_e)
    d_hat = iso.predict(d_o)
    return float(np.sqrt(((d_e - d_hat) ** 2).sum() / (d_e ** 2).sum()))


X_scaled = StandardScaler().fit_transform(weighted_matrix)
D = pairwise_distances(X_scaled, metric="euclidean")

pca = PCA(n_components=2, random_state=SEED)
pca_ind = pca.fit_transform(X_scaled)

trend_scaled = StandardScaler().fit_transform(trend_matrix)
pca_time = PCA(n_components=2, random_state=SEED)
pca_time_coords = pca_time.fit_transform(trend_scaled)

nmds_coords = run_mds(D, metric=False)
gnmds_coords = run_mds(D, metric=True)

ordination = {
    "pca_indicator_pc1_var": round(float(pca.explained_variance_ratio_[0]), 4),
    "pca_indicator_pc2_var": round(float(pca.explained_variance_ratio_[1]), 4),
    "pca_temporal_pc1_var": round(float(pca_time.explained_variance_ratio_[0]), 4),
    "pca_temporal_pc2_var": round(float(pca_time.explained_variance_ratio_[1]), 4),
    "nmds_stress1": round(kruskal_stress1(D, nmds_coords), 4),
    "gnmds_stress1": round(kruskal_stress1(D, gnmds_coords), 4),
    "n_points": int(len(weighted_matrix)),
}
for k, v in ordination.items():
    print(f"  {k:24s} {v}")
print("\n  Note: 12 points in 2 dimensions. Low stress is close to guaranteed here.")
save_checkpoint("stage7_ordination", ordination)

---
# Stage 8 — PERMANOVA

Anderson's (2001) pseudo-F with a label permutation test, checking whether
between-lens variance exceeds within-lens variance.

**The test runs at two units of analysis, and they answer different questions.**

| Unit | n | Question |
|---|---|---|
| Indicator | 12 | Do the 12 indicators separate into 3 lens groups? |
| Article | thousands | Do individual articles separate by lens? |

Expect very different numbers. With only 12 points and 3 groups the indicator-level
test has little power, and F values around 1–3 are normal there even when the grouping
is sound — on the paper's own published Figure 3b matrix it returns F ≈ 1.9. The
article-level test, run on thousands of records, returns F in the tens or hundreds.
The paper reports **F > 25, p < 0.001**, which corresponds to the article-level unit.

Both are run below so you can see which number your data produces at which unit, and
report n, unit and distance measure alongside any F you quote.

Article-level PERMANOVA on the full corpus would need an n × n distance matrix
permuted thousands of times, which is not practical at 5,634 records. It is run on
repeated subsamples instead and the spread across them is reported.

In [ ]:
def permanova(distances, groups, permutations=9999, seed=SEED):
    """Anderson (2001) pseudo-F permutation test on a square distance matrix."""
    D_ = np.asarray(distances, dtype=float)
    g = np.asarray(groups)
    n, labels = D_.shape[0], np.unique(groups)
    a = len(labels)
    if a < 2:
        raise ValueError("PERMANOVA needs at least two groups.")
    if n - a < 1:
        raise ValueError(f"n={n} is too small for {a} groups.")

    D2 = D_ ** 2
    ss_total = D2.sum() / (2 * n)

    def within(vec):
        s = 0.0
        for lab in labels:
            idx = np.flatnonzero(vec == lab)
            if len(idx) > 1:
                s += D2[np.ix_(idx, idx)].sum() / (2 * len(idx))
        return s

    ss_w = within(g)
    f_obs = ((ss_total - ss_w) / (a - 1)) / (ss_w / (n - a)) if ss_w > 0 else np.inf

    rng = np.random.default_rng(seed)
    perm, count = g.copy(), 0
    for _ in range(permutations):
        rng.shuffle(perm)
        w = within(perm)
        f = ((ss_total - w) / (a - 1)) / (w / (n - a)) if w > 0 else np.inf
        count += f >= f_obs
    return {"F": round(float(f_obs), 4),
            "p_value": round(float((count + 1) / (permutations + 1)), 5),
            "R2": round(float((ss_total - ss_w) / ss_total), 4),
            "n": int(n), "groups": int(a), "permutations": int(permutations)}


def permanova_articles(sims, groups, max_n=800, repeats=5, permutations=999, seed=SEED):
    """Article-level PERMANOVA over repeated subsamples."""
    raw = np.asarray(sims, dtype=float)
    tot = raw.sum(axis=1, keepdims=True)
    share = np.divide(raw, tot, out=np.zeros_like(raw), where=tot > 0)
    g, rng = np.asarray(groups), np.random.default_rng(seed)

    out = []
    for r in range(repeats):
        idx = rng.choice(len(share), size=min(max_n, len(share)), replace=False)
        res = permanova(pairwise_distances(share[idx]), g[idx],
                        permutations=permutations, seed=seed + r)
        res["subsample"] = r + 1
        out.append(res)
    return pd.DataFrame(out)[["subsample", "n", "F", "p_value", "R2"]]

In [ ]:
groups_ind = np.array([INDICATOR_TO_LENS[i] for i in weighted_matrix.index])

print("=" * 66)
print("A. INDICATOR LEVEL (n = 12) - your rebuilt matrix")
print("=" * 66)
pm_ind = permanova(D, groups_ind, permutations=CONFIG["n_permutations"])
for k, v in pm_ind.items():
    print(f"  {k:14s} {v}")

print("\n" + "=" * 66)
print("B. INDICATOR LEVEL (n = 12) - the paper's OWN published Figure 3b matrix")
print("=" * 66)
D_paper = pairwise_distances(StandardScaler().fit_transform(PAPER_LENS_WEIGHTS))
pm_paper = permanova(D_paper, groups_ind, permutations=CONFIG["n_permutations"])
for k, v in pm_paper.items():
    print(f"  {k:14s} {v}")
print("\n  This uses the paper's published numbers, not the demo corpus.")
print("  It does not reproduce F > 25 at this unit of analysis.")

print("\n" + "=" * 66)
print(f"C. ARTICLE LEVEL (subsamples of n = {CONFIG['permanova_max_n']})")
print("=" * 66)
pm_art = permanova_articles(sims[LENSES], corpus["Keyword_Lens"].to_numpy(),
                            max_n=CONFIG["permanova_max_n"],
                            repeats=CONFIG["permanova_repeats"], permutations=999)
display(pm_art)
print(f"  Median F across subsamples: {pm_art['F'].median():.2f}")
print("  This is the unit that produces F in the tens.")

save_checkpoint("stage8_permanova", pd.DataFrame(
    [{"unit": "indicator (rebuilt)", **pm_ind},
     {"unit": "indicator (paper Fig 3b)", **pm_paper},
     {"unit": f"article (median of {CONFIG['permanova_repeats']} subsamples)",
      "F": round(float(pm_art["F"].median()), 4),
      "p_value": round(float(pm_art["p_value"].median()), 5),
      "R2": round(float(pm_art["R2"].median()), 4),
      "n": int(pm_art["n"].iloc[0]), "groups": 3, "permutations": 999}]), index=False)

---
# Stage 9 — Figures

Figures 2 to 6, each computed from the corpus rather than from typed-in constants.
In `demo` mode every figure carries a stamp so a demo output can never be mistaken
for a paper result.

Each figure is saved to `outputs/figures/` at the DPI set in `CONFIG`.

In [ ]:
def stamp(fig):
    """Mark demo-mode figures so they cannot be mistaken for paper results."""
    if CONFIG["data_mode"] == "demo":
        fig.text(0.5, 0.5, "DEMO DATA\nNOT PAPER RESULTS",
                 fontsize=44, color="grey", alpha=0.13, ha="center", va="center",
                 rotation=30, zorder=1000, weight="bold")
    return fig


def finish(fig, name):
    stamp(fig)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path)
    plt.show()
    print(f"  saved: {path}")


def lowess_fit(y, x, frac=0.4):
    """statsmodels LOWESS when available, local linear regression otherwise."""
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        s = lowess(np.asarray(y, float), np.asarray(x, float), frac=frac,
                   return_sorted=True)
        return s[:, 0], s[:, 1]
    except ImportError:
        x, y = np.asarray(x, float), np.asarray(y, float)
        o = np.argsort(x); x, y = x[o], y[o]
        n = len(x); r = max(2, int(np.ceil(frac * n)))
        out = np.empty(n)
        X = np.column_stack([np.ones(n), x])
        for i in range(n):
            d = np.abs(x - x[i])
            h = np.sort(d)[r - 1] or 1.0
            w = (1 - np.clip(d / h, 0, 1) ** 3) ** 3
            W = np.diag(w)
            try:
                b = np.linalg.lstsq(X.T @ W @ X, X.T @ W @ y, rcond=None)[0]
                out[i] = b[0] + b[1] * x[i]
            except np.linalg.LinAlgError:
                out[i] = y[i]
        return x, out

In [ ]:
# ---------------------------------------------------- Figure 2
annual = corpus.groupby(["Year", "SearchIndicator"]).size().reset_index(name="Count")
bins = [1989, 2012, 2015, 2018, 2020, 2022, 2026]
plabels = ["<=2012", "2013-15", "2016-18", "2019-20", "2021-22", "2023-25"]
corpus["Period"] = pd.cut(corpus["Year"], bins=bins, labels=plabels)
period_counts = corpus.groupby(["Period", "SearchIndicator"], observed=True) \
                      .size().reset_index(name="Count")

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
palette = dict(zip(INDICATORS, sns.color_palette("tab20", len(INDICATORS))))

ax = axes[0, 0]
for ind, grp in annual.groupby("SearchIndicator"):
    ax.plot(grp["Year"], grp["Count"], marker="o", ms=3, lw=1.4,
            color=palette[ind], label=ind)
ax.set(title="a) Annual publications per indicator", xlabel="Year", ylabel="Papers")
ax.legend(fontsize=6, ncol=2)

# Each (Period, Indicator) pair holds exactly one count, so splitting the boxplot by
# hue="Period" would draw a box over a single point. Box each indicator across its
# periods instead, with the individual periods overlaid as points.
ax = axes[0, 1]
order = period_counts.groupby("SearchIndicator")["Count"].median()                      .sort_values(ascending=False).index
sns.boxplot(data=period_counts, x="SearchIndicator", y="Count", order=order,
            ax=ax, color="#D6EAF8", width=0.6, fliersize=0)
sns.stripplot(data=period_counts, x="SearchIndicator", y="Count", hue="Period",
              order=order, ax=ax, palette="Set2", size=5, jitter=0.18,
              edgecolor="grey", linewidth=0.4)
ax.set(title="b) Distribution of indicator frequency across periods",
       xlabel="", ylabel="Frequency")
ax.tick_params(axis="x", rotation=60)
ax.legend(fontsize=6, title="Period", title_fontsize=7, loc="upper right")

# Plot on the categorical period order, not on whatever order groupby returns.
ax = axes[1, 0]
for ind, grp in period_counts.groupby("SearchIndicator"):
    grp = grp.set_index("Period").reindex(plabels).reset_index()
    ax.plot(range(len(plabels)), grp["Count"].values, marker="o", ms=4, lw=1.4,
            color=palette[ind], label=ind)
ax.set_xticks(range(len(plabels)))
ax.set_xticklabels(plabels)
ax.set(title="c) Trajectories across periods", xlabel="Period", ylabel="Frequency")
ax.tick_params(axis="x", rotation=30)
ax.legend(fontsize=6, ncol=2)

# Panel d: growth measured by linear slope over years present, not first-vs-last
ax = axes[1, 1]
summary = annual.groupby("SearchIndicator")["Count"].sum().to_frame("TotalPubs")
slopes = {}
for ind, grp in annual.groupby("SearchIndicator"):
    if len(grp) >= 3:
        slopes[ind] = np.polyfit(grp["Year"], grp["Count"], 1)[0]
    else:
        slopes[ind] = 0.0
summary["Growth"] = pd.Series(slopes)
mt, mg = summary["TotalPubs"].median(), summary["Growth"].median()

ax.scatter(summary["TotalPubs"], summary["Growth"],
           s=np.clip(summary["TotalPubs"] / 2, 40, 900),
           c=[palette[i] for i in summary.index], alpha=0.75, edgecolor="white", lw=1.5)
for ind, row in summary.iterrows():
    ax.annotate(ind, (row["TotalPubs"], row["Growth"]), fontsize=7,
                xytext=(6, 5), textcoords="offset points")
ax.axvline(mt, color="grey", ls="--", alpha=0.6)
ax.axhline(mg, color="grey", ls="--", alpha=0.6)
xr = summary["TotalPubs"].max() - summary["TotalPubs"].min()
yr_ = summary["Growth"].max() - summary["Growth"].min()
for lbl, dx, dy in [("Dominant", .28, .38), ("Emerging", -.42, .38),
                    ("Marginal", -.42, -.42), ("Declining", .28, -.42)]:
    ax.text(mt + dx * xr, mg + dy * yr_, lbl, fontsize=9, color="grey", weight="bold")
ax.set(title="d) Dominant / Emerging / Declining / Marginal",
       xlabel="Total publications", ylabel="Growth (papers per year, OLS slope)")

fig.suptitle("Figure 2 - Trends and growth dynamics of search indicators",
             fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure2_indicator_trends")

In [ ]:
# ---------------------------------------------------- Figure 3
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

ax = axes[0]
sns.heatmap(weighted_matrix, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Semantic weight"})
ax.set(title="b) Weighted semantic matches", xlabel="", ylabel="Search indicator")
ax.set_xticklabels([LENS_SHORT[t.get_text()] for t in ax.get_xticklabels()], rotation=0)

ax = axes[1]
G = nx.Graph()
for lens in LENSES:
    G.add_node(lens, kind="lens")
for ind in weighted_matrix.index:
    G.add_node(ind, kind="indicator")
    for lens in LENSES:
        w = float(weighted_matrix.loc[ind, lens])
        if w > 0:
            G.add_edge(ind, lens, weight=w)

pos = nx.spring_layout(G, seed=SEED, k=1.6, iterations=200)
maxw = max(d["weight"] for *_, d in G.edges(data=True))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#95a5a6",
                       width=[0.3 + 3.5 * d["weight"] / maxw for *_, d in G.edges(data=True)],
                       alpha=0.55)
nx.draw_networkx_nodes(G, pos, ax=ax,
                       nodelist=LENSES, node_color=[LENS_COLORS[l] for l in LENSES],
                       node_size=1500, edgecolors="white", linewidths=2)
nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=list(weighted_matrix.index),
                       node_color="#D5DBDB", node_size=650,
                       edgecolors="#7F8C8D", linewidths=1)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=6,
                        labels={n: LENS_SHORT.get(n, n) for n in G.nodes()})
ax.set(title="c) Semantic network")
ax.axis("off")

ax = axes[2]
prop = weighted_matrix.div(weighted_matrix.sum(axis=1), axis=0)
prop.plot(kind="bar", stacked=True, ax=ax,
          color=[LENS_COLORS[l] for l in LENSES], width=0.8, legend=False)
ax.set(title="d) Proportional lens contribution", xlabel="", ylabel="Proportion", ylim=(0, 1))
ax.tick_params(axis="x", rotation=60)
ax.legend([LENS_SHORT[l] for l in LENSES], fontsize=7, loc="lower right", framealpha=0.9)

fig.suptitle("Figure 3 - Semantic mapping of conceptual lenses",
             fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure3_semantic_mapping")

In [ ]:
# ---------------------------------------------------- Figure 4
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for lens, marker in zip(LENSES, ["o", "s", "^"]):
    c = LENS_COLORS[lens]
    ax.scatter(trend_matrix.index, trend_matrix[lens], color=c, marker=marker,
               s=28, alpha=0.45)
    xs, ys = lowess_fit(trend_matrix[lens].values, trend_matrix.index.values, frac=0.4)
    ax.plot(xs, ys, color=c, lw=2.5, label=lens)
ax.set(title="a) Lens trajectories (LOWESS smoothed) - your corpus",
       xlabel="Year", ylabel="Weighted contribution")
ax.legend(fontsize=8)

ax = axes[1]
for lens in LENSES:
    xs, ys = lowess_fit(PAPER_TEMPORAL[lens].values, PAPER_TEMPORAL.index.values, frac=0.4)
    ax.plot(xs, ys, color=LENS_COLORS[lens], lw=2.5, ls="--", label=lens)
    ax.scatter(PAPER_TEMPORAL.index, PAPER_TEMPORAL[lens],
               color=LENS_COLORS[lens], s=28, alpha=0.45)
ax.set(title="b) The paper's published values (Figure 4b), for comparison",
       xlabel="Year", ylabel="Weighted contribution")
ax.legend(fontsize=8)

fig.suptitle("Figure 4 - Temporal evolution of conceptual lenses",
             fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure4_temporal_evolution")

In [ ]:
# ---------------------------------------------------- Figure 5
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
handles = [mpatches.Patch(color=c, label=LENS_SHORT[l]) for l, c in LENS_COLORS.items()]
dominant = weighted_matrix.idxmax(axis=1)

ax = axes[0, 0]
years = trend_matrix.index.to_numpy()
sc = ax.scatter(pca_time_coords[:, 0], pca_time_coords[:, 1], c=years,
                cmap="viridis", s=90, zorder=3, edgecolor="white", lw=1.2)
ax.plot(pca_time_coords[:, 0], pca_time_coords[:, 1], color="grey", lw=0.8,
        alpha=0.5, zorder=2)
for i, y in enumerate(years):   # alternate the offset so dense runs stay readable
    ax.annotate(str(y), (pca_time_coords[i, 0], pca_time_coords[i, 1]), fontsize=6.5,
                xytext=(5, 6 if i % 2 == 0 else -11), textcoords="offset points")
for j, lens in enumerate(LENSES):
    v = pca_time.components_[:, j] * np.abs(pca_time_coords).max() * 0.85
    ax.arrow(0, 0, v[0], v[1], color=LENS_COLORS[lens], width=0.012,
             head_width=0.13, alpha=0.85, zorder=4)
    ax.text(v[0] * 1.08, v[1] * 1.08, LENS_SHORT[lens], color=LENS_COLORS[lens],
            fontsize=8, weight="bold")
plt.colorbar(sc, ax=ax, label="Year", fraction=0.045)
ax.axhline(0, color="grey", lw=0.5, ls="--"); ax.axvline(0, color="grey", lw=0.5, ls="--")
ax.set(title=f"a) PCA biplot - temporal trajectory",
       xlabel=f"PC1 ({pca_time.explained_variance_ratio_[0]:.1%})",
       ylabel=f"PC2 ({pca_time.explained_variance_ratio_[1]:.1%})")

for ax, coords, label, stress in [
        (axes[0, 1], nmds_coords, "b) NMDS of indicators", ordination["nmds_stress1"]),
        (axes[1, 1], gnmds_coords, "d) GNMDS (metric)", ordination["gnmds_stress1"])]:
    for i, ind in enumerate(weighted_matrix.index):
        ax.scatter(coords[i, 0], coords[i, 1], color=LENS_COLORS[dominant[ind]],
                   s=110, zorder=3, edgecolor="white", lw=1.2)
        ax.annotate(ind, (coords[i, 0], coords[i, 1]), fontsize=7,
                    xytext=(6, 4), textcoords="offset points")
    ax.legend(handles=handles, fontsize=7, loc="best")
    ax.set(title=f"{label} - Kruskal stress-1 = {stress:.3f}",
           xlabel="Dimension 1", ylabel="Dimension 2")

ax = axes[1, 0]
for i, ind in enumerate(weighted_matrix.index):
    ax.scatter(nmds_coords[i, 0], nmds_coords[i, 1],
               color=LENS_COLORS[groups_ind[i]], s=110, zorder=3,
               edgecolor="white", lw=1.2)
    ax.annotate(ind, (nmds_coords[i, 0], nmds_coords[i, 1]), fontsize=7,
                xytext=(6, 4), textcoords="offset points")
for lens in LENSES:
    idx = np.flatnonzero(groups_ind == lens)
    if len(idx) < 3:
        continue
    pts = nmds_coords[idx]
    vals, vecs = np.linalg.eigh(np.cov(pts.T))
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    ax.add_patch(Ellipse(pts.mean(axis=0),
                         width=2 * 1.96 * np.sqrt(max(vals[0], 1e-9)),
                         height=2 * 1.96 * np.sqrt(max(vals[1], 1e-9)),
                         angle=np.degrees(np.arctan2(*vecs[:, 0][::-1])),
                         edgecolor=LENS_COLORS[lens], facecolor=LENS_COLORS[lens],
                         alpha=0.10, lw=1.8, ls="--"))
ax.legend(handles=handles, fontsize=7, loc="best")
ax.set(title=f"c) NMDS with a-priori lens groups\n"
             f"PERMANOVA F={pm_ind['F']:.2f}, p={pm_ind['p_value']:.3f} (n=12)",
       xlabel="NMDS Dimension 1", ylabel="NMDS Dimension 2")

fig.suptitle("Figure 5 - Ordination of conceptual lenses", fontsize=13, weight="bold")
fig.tight_layout()
finish(fig, "figure5_ordination")

In [ ]:
# ---------------------------------------------------- Figure 6
# Subsystem x lens influence, derived from the weighted matrix: each subsystem
# inherits the mean semantic weight of the indicators that feed it (see Stage 1).
subsystem_matrix = pd.DataFrame(
    {lens: {sub: weighted_matrix.loc[inds, lens].mean()
            for sub, inds in SUBSYSTEM_INDICATORS.items()} for lens in LENSES}
).reindex(SUBSYSTEMS)
subsystem_matrix.index.name = "Subsystem"
save_checkpoint("stage9_subsystem_matrix", subsystem_matrix)

G = nx.Graph()
for lens in LENSES:
    G.add_node(lens, kind="lens")
for sub in SUBSYSTEMS:
    G.add_node(sub, kind="subsystem")
    for lens in LENSES:
        w = float(subsystem_matrix.loc[sub, lens])
        if w > 0:
            G.add_edge(sub, lens, weight=w)

pos = {}
for i, sub in enumerate(SUBSYSTEMS):
    a = 2 * np.pi * i / len(SUBSYSTEMS) + np.pi / 12
    pos[sub] = (np.cos(a) * 3.4, np.sin(a) * 3.4)
pos["Key Stakeholders and entities"] = (0.85, 0.5)
pos["Models of Interaction"] = (-0.85, 0.5)
pos["External influencing factors"] = (0.0, -0.95)

fig, ax = plt.subplots(figsize=(14, 12))
maxw = max(d["weight"] for *_, d in G.edges(data=True))
for u, v, d in G.edges(data=True):
    lens = u if u in LENS_COLORS else v
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
            color=LENS_COLORS[lens], lw=0.4 + 5.5 * d["weight"] / maxw,
            alpha=0.45, zorder=1, solid_capstyle="round")

nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=SUBSYSTEMS, node_color="#FCF3CF",
                       node_size=2600, edgecolors="#B7950B", linewidths=1.6)
nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=LENSES,
                       node_color=[LENS_COLORS[l] for l in LENSES],
                       node_size=3400, edgecolors="white", linewidths=2.5)
for n, (x, y) in pos.items():
    ax.text(x, y, "\n".join(LENS_SHORT.get(n, n).split()), ha="center", va="center",
            fontsize=7, weight="bold",
            color="white" if n in LENS_COLORS else "#4A3B00", zorder=5)

ax.legend(handles=[mpatches.Patch(color=c, label=l) for l, c in LENS_COLORS.items()]
          + [mpatches.Patch(facecolor="#FCF3CF", edgecolor="#B7950B",
                            label="Urban governance subsystem")],
          fontsize=8, loc="lower right", framealpha=0.95)
ax.set_title("Figure 6 - Systems-based governance framework of AI Urbanism\n"
             "Edge width = weighted strength of each lens's influence",
             fontsize=13, weight="bold")
ax.axis("off")
ax.margins(0.13)
finish(fig, "figure6_governance_framework")
display(subsystem_matrix.round(2))

---
# Stage 10 — Reproducibility report

A single manifest recording what ran, with what settings, in what environment, and
what came out — including a SHA-256 of every output so a later run can be compared
byte for byte.

In [ ]:
import hashlib

def sha256(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


outputs = sorted(p for p in OUT_DIR.rglob("*") if p.is_file())
manifest = {
    "paper": {"title": "Governing with artificial intelligence: Mapping the "
                       "knowledge systems shaping urban intelligence",
              "authors": "Lartey, D. & Law, K.M.Y.",
              "journal": "Technology in Society 86 (2026) 103321",
              "doi": "10.1016/j.techsoc.2026.103321"},
    "run": {"data_mode": CONFIG["data_mode"], "backend": BACKEND,
            "n_articles": int(len(corpus)), "seed": SEED,
            "threshold": CONFIG["threshold"], "threshold_on": CONFIG["threshold_on"],
            "keyword_weight": CONFIG["keyword_weight"]},
    "environment": ENV,
    "results": {"ordination": ordination,
                "permanova_indicator_rebuilt": pm_ind,
                "permanova_indicator_paper_matrix": pm_paper,
                "permanova_article_median_F": round(float(pm_art["F"].median()), 3),
                "match_to_published_fig3b": fit,
                "validation": validation_scores},
    "outputs": {str(p.relative_to(OUT_DIR)): {"bytes": p.stat().st_size,
                                              "sha256_16": sha256(p)} for p in outputs},
}

path = OUT_DIR / "reproducibility_report.json"
path.write_text(json.dumps(manifest, indent=2, default=str))

print("=" * 68)
print("RUN COMPLETE")
print("=" * 68)
print(f"  mode         {CONFIG['data_mode']}")
print(f"  backend      {BACKEND}")
print(f"  articles     {len(corpus):,}")
print(f"  outputs      {len(outputs)} files in {OUT_DIR.resolve()}")
print(f"  manifest     {path.name}")
if CONFIG["data_mode"] == "demo":
    print("\n  Reminder: demo mode. These numbers do not reproduce the paper.")
print("=" * 68)
for p in outputs:
    print(f"  {str(p.relative_to(OUT_DIR)):48s} {p.stat().st_size / 1024:8.1f} KB")

---

### Citation

```bibtex
@article{lartey2026governing,
  title   = {Governing with artificial intelligence: Mapping the knowledge
             systems shaping urban intelligence},
  author  = {Lartey, Desmond and Law, Kris M. Y.},
  journal = {Technology in Society},
  volume  = {86},
  pages   = {103321},
  year    = {2026},
  doi     = {10.1016/j.techsoc.2026.103321}
}
```

Questions: **larteydesmond3@gmail.com**